In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import os
from os import listdir
import matplotlib.lines as mlines
from matplotlib.patches import Patch
import scipy
from scipy.stats import pearsonr
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import StratifiedKFold
import time
import pickle
import pandas as pd
# plt.rcParams["font.family"] = "Helvetica"

## 1. Functions

In [ ]:
def count_number(labels):
    '''
    Inputs
    ----------
    labels: shape=(2,trial_num) 
    - rows represent t1, t2 respectively
    
    Returns
    ----------
    all_label_number: shape=(2,4) 
    - rows represent t1, t2
    - columns represent trial number for each location 0, 1, 2, 3
    
    '''
    all_label_num = []
    for label in labels:
        each_label_num = []
        # count the number of trials that each class of stimuli or rewards has
        for i in range(4):
            each_label_num.append(np.sum(label == i))
        all_label_num.append(np.array(each_label_num).reshape(1,-1))
    return np.concatenate(all_label_num, axis=0)


def create_pseudo_popu(labels_cat, units_cat, min_repeated_trials):
    """
    Create balanced cross-session pseudopopulations for decoding Target 1 and
    Target 2 locations.

    For each session, the function samples the same number of trials from each
    of the four locations for the target being decoded. Neural activity is then
    concatenated across sessions. The location labels of the other target are
    also saved for each unit.

    Inputs
    ----------
    labels_cat : list
        One label array per session.
        labels_cat[i][0] contains Target 1 labels and labels_cat[i][1]
        contains Target 2 labels.

    units_cat : list
        Neural activity arrays for each session with shape
        (n_trials, n_units, ...).

    min_repeated_trials : int
        Number of trials sampled from each of the four location classes.

    Returns
    -------
    feature : dict
        Pseudopopulation activity for Target 1 and Target 2 decoding.

    label : dict
        Contains the balanced decoded-target labels and the unmatched
        other-target labels for each unit.
    """
    
    feature = {'t1': [], 't2': []}
    matched_label = np.array([0]*min_repeated_trials + [1]*min_repeated_trials + [2]*min_repeated_trials + [3]*min_repeated_trials)
    label = {'matched_label': matched_label, 't1_unmatched_label': [], 't2_unmatched_label': []}

    for i, labels in enumerate(labels_cat):
        for j in range(2):
            sel_units = []
            sel_unmatched_label = []
            for k in range(4):
                class_idx = np.where(labels[j] == k)[0]
                sel_class_idx = np.random.choice(class_idx, min_repeated_trials, replace=False)
                sel_units.append(units_cat[i][sel_class_idx])
                if j == 0:
                    sel_unmatched_label.append(labels[1][sel_class_idx])
                if j == 1:
                    sel_unmatched_label.append(labels[0][sel_class_idx])
            
            sel_units = np.vstack(sel_units)
            sel_unmatched_label = np.hstack(sel_unmatched_label)
            
            if j == 0:
                feature['t1'].append(sel_units)
                label['t1_unmatched_label'].append(np.tile(sel_unmatched_label, (units_cat[i].shape[1], 1)))
            if j == 1:
                feature['t2'].append(sel_units)
                label['t2_unmatched_label'].append(np.tile(sel_unmatched_label, (units_cat[i].shape[1], 1)))
    
    for key, value in feature.items():
        feature[key] = np.concatenate(value, axis=1)
    label['t1_unmatched_label'] = np.vstack(label['t1_unmatched_label'])
    label['t2_unmatched_label'] = np.vstack(label['t2_unmatched_label'])
    
    return feature, label

def t_to_idx(t):
    return int((t-(-2))/0.05)

def normalization(X1, X2, labels):
    unique_labels = np.unique(labels)
    mean_responses = []
    
    for l in unique_labels:
        idx = (labels == l)
        mean_responses.append(np.mean(X1[idx], axis=0).T)
        mean_responses.append(np.mean(X2[idx], axis=0).T)
    
    mean_responses = np.vstack(mean_responses)
    
    X_min = np.min(mean_responses, axis=0)[np.newaxis,:,np.newaxis]
    X_max = np.max(mean_responses, axis=0)[np.newaxis,:,np.newaxis]
    X_range = X_max - X_min
    X_range[X_range==0] = 1
    
    # normalized X1 and X2
    normalized_X1 = (X1 - X_min)/X_range
    normalized_X2 = (X2 - X_min)/X_range
      
    return normalized_X1, normalized_X2

## 2. Concatenate trials across sessions

In [ ]:
## info
trial_type = 'correct'
subject = 'Tir'

# concatenate units and labels for two conditions (retain & update)
cond1_units_cat = []
cond2_units_cat = []
cond1_labels_cat = []
cond2_labels_cat = []
area_all_units = {}


areas = ['PFC']
for area in areas:
    area_all_units[area] = []

min_repeated_trials = None
fixed_min = 15 # each session has at least 15 correct trials per location

path = f'/Users/huidili/Desktop/wmupdate/analysis_data/{trial_type}_trials/'
area_path = path+'area_df/'

for filename in listdir(area_path):
    if ('area' in filename):
        if 'ISA' in filename:
            sub_name, session_num, _, _ = filename.split('_')
            session_name = sub_name+'_'+session_num 
        elif 'Tir' in filename:
            session_name = filename.split('_')[0]

        if (subject == 'Tir' and 'Tir' in session_name) or (subject == 'ISA' and 'ISA' in session_name):            
            condition_path = path+'condition_df/'
            spk_path = path+'spike_rate/150bin_50step/'
            label_path = path+'labels/'

            area_df = pd.read_pickle(area_path+filename)  
            condition_df = pd.read_pickle(condition_path+session_name+'_condition_df.pkl')
            spk_data = np.load(spk_path+session_name+'_spike_rate.npz')
            label_data = np.load(label_path+session_name+'_labels.npy')
            cond1_trial_idx = condition_df['retain']
            cond2_trial_idx = condition_df['update']
            cond1_labels = label_data[:, cond1_trial_idx]
            cond2_labels = label_data[:, cond2_trial_idx]
            min_labels = min(np.min(count_number(cond1_labels[:2])), np.min(count_number(cond2_labels[:2]))) 

            if min_labels >= fixed_min: 
                if min_repeated_trials is None or (min_labels < min_repeated_trials):
                    min_repeated_trials = min_labels

                for area in areas:
                    area_idx = area_df[area]
                    area_all_units[area].append(area_idx)
                cond1_units_cat.append(spk_data['spike_rate'][cond1_trial_idx])
                cond2_units_cat.append(spk_data['spike_rate'][cond2_trial_idx])
                cond1_labels_cat.append(cond1_labels)
                cond2_labels_cat.append(cond2_labels)

for area in areas:
    area_all_units[area] = np.concatenate(area_all_units[area])

In [ ]:
f'Number of included sessions: {len(cond1_units_cat)}'

In [ ]:
f'Number of trials per location: {min_repeated_trials}'

## 2. Subspace alignment

In [ ]:
# lasso regression CV for selecting optimal alpha
alpha_grid = 10.0**(-np.arange(6))

matched_labels = np.repeat(np.arange(4), min_repeated_trials)
n_splits = min(5, min_repeated_trials)

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=0
)

cv_splits = list(
    cv.split(
        np.zeros(len(matched_labels)),
        matched_labels
    )
)

In [ ]:
corr_dict = {}

chunk_size = 10
prev_iteration = 0
end_iteration = 1000

start_t = 1.8
stop_t = 3.2
n_tbins = t_to_idx(stop_t) - t_to_idx(start_t)

target_names = ['chosen_t1', 'unchosen_t2', 'unchosen_t1', 'chosen_t2']
target_idx_dict = {'first':(0,2), 'second':(1,3)}
# target_idx_dict = {'high_rwd':(0,3), 'low_rwd':(1,2)}


for iteration in range(prev_iteration, end_iteration):   
    # create pseudopopulation for each condition
    cond1_feature, cond1_label = create_pseudo_popu(cond1_labels_cat, cond1_units_cat, min_repeated_trials)
    cond2_feature, cond2_label = create_pseudo_popu(cond2_labels_cat, cond2_units_cat, min_repeated_trials)

    # separate units by brain regions
    for area in areas:
        area_idx = area_all_units[area]
        
        if iteration == prev_iteration:
            for cond in target_idx_dict.keys():
                corr_dict[f'{area}_{cond}'] = np.zeros((chunk_size, n_tbins))
        
        # 2 targets in cond1
        chosen_t1 = cond1_feature['t1'][:,area_idx,t_to_idx(start_t):t_to_idx(stop_t)]
        unchosen_t2 = cond1_feature['t2'][:,area_idx,t_to_idx(start_t):t_to_idx(stop_t)]
        # 2 targets in cond2
        unchosen_t1 = cond2_feature['t1'][:,area_idx,t_to_idx(start_t):t_to_idx(stop_t)]
        chosen_t2 = cond2_feature['t2'][:,area_idx,t_to_idx(start_t):t_to_idx(stop_t)]
        
        # subspace correlation
        all_popu_resp = [chosen_t1, unchosen_t2, unchosen_t1, chosen_t2]
        all_unmatched_label = [cond1_label['t1_unmatched_label'][area_idx], cond1_label['t2_unmatched_label'][area_idx], cond2_label['t1_unmatched_label'][area_idx], cond2_label['t2_unmatched_label'][area_idx]]
        
        for cond, idx in target_idx_dict.items():
            resp1 = all_popu_resp[idx[0]]
            resp2 = all_popu_resp[idx[1]]
            unmatched_label1 = all_unmatched_label[idx[0]]
            unmatched_label2 = all_unmatched_label[idx[1]]
            normalized_resp1, normalized_resp2 = normalization(resp1, resp2, cond1_label['matched_label'])
            
            for t in range(n_tbins):
                resp1_t = normalized_resp1[:,:,t]
                resp2_t = normalized_resp2[:,:,t]
                
                coef1_t = np.zeros((4, resp1_t.shape[1]))
                coef2_t = np.zeros((4, resp2_t.shape[1]))

                for n in range(resp1_t.shape[1]):
                    y1_reg = resp1_t[:, n]
                    y2_reg = resp2_t[:, n]

                    X1_reg = np.zeros((len(y1_reg), 8))
                    X2_reg = np.zeros((len(y2_reg), 8))

                    for m in range(4):
                        # First four columns: location of the target being analyzed.
                        X1_reg[cond1_label['matched_label'] == m, m] = 1
                        X2_reg[cond1_label['matched_label'] == m, m] = 1

                        # Last four columns: location of the other target.
                        X1_reg[unmatched_label1[n] == m, m + 4] = 1
                        X2_reg[unmatched_label2[n] == m, m + 4] = 1

                    # Choose alpha separately for each neuron and response.
                    reg1 = LassoCV(
                        alphas=alpha_grid,
                        cv=cv_splits,
                        max_iter=5000,
                        fit_intercept=False,
                        n_jobs=-1
                    ).fit(X1_reg, y1_reg)

                    reg2 = LassoCV(
                        alphas=alpha_grid,
                        cv=cv_splits,
                        max_iter=5000,
                        fit_intercept=False,
                        n_jobs=-1
                    ).fit(X2_reg, y2_reg)

                    coef1_t[:, n] = reg1.coef_[:4]
                    coef2_t[:, n] = reg2.coef_[:4]

                # subtracted by condition-mean
                coef1_t -= np.mean(coef1_t, axis=0)
                coef2_t -= np.mean(coef2_t, axis=0)
                
                # compute correlation
                corr_t = []
                for i in range(4):
                    corr_t.append(pearsonr(coef1_t[i], coef2_t[i])[0])
        

                corr_dict[f'{area}_{cond}'][int(iteration%chunk_size),t] = np.mean(corr_t)

    
    if (iteration+1)%chunk_size == 0:
        for area in areas:
            for cond in target_idx_dict.keys():
                path = f'/Users/huidili/Desktop/wmupdate/analysis_result/subspace_alignment/1000resamples/{area}_{cond}/{subject}/'
                np.save(path+f'iteration{iteration+1}.npy', corr_dict[f'{area}_{cond}'])
        
        for cond in target_idx_dict.keys():
            corr_dict[f'{area}_{cond}'] = np.zeros((chunk_size, n_tbins))
        print(f'iteration: {iteration+1}')